# Notebook completo — modelo preditivo de risco de defasagem

**Objetivo:** documentar o fluxo desde a **base bruta** até a **avaliação** do modelo que estima o risco de o estudante estar **em defasagem** (`defasagem` negativa na base harmonizada PEDE mesma regra que `defasagem < 0` no código).

**Roteiro:** (1) configuração e carga da base; (2) limpeza e unificação PEDE; (3) análise exploratória (EDA); (4) feature engineering; (5) treino/teste; (6) modelagem; (7) avaliação; (8) **principais insights** (texto + gráficos para slides); (9) exportação opcional do `joblib`.

**Fonte da lógica (mantida nos `.py`):** limpeza em `src/pede_cleaning.py`; modelo em `src/pede_model.py` (também usado pelo Streamlit e por `scripts/train_model.py`). O notebook **importa** esses módulos em vez de duplicar código.

**Como executar:** execute as células **em ordem**. A primeira célula de código define `ROOT` e `sys.path`. Se pular a **seção 2**, o `df` vem só do parquet da **seção 1** (recomendado que o parquet já exista e esteja atualizado); se rodar a **seção 2**, o `df` é recalculado a partir dos CSVs (`build_unified` em `src/pede_cleaning.py`). A **seção 3 (EDA)** usa o `df` carregado em 1 ou regenerado em 2.


## 1. Configuração e base de dados

- Definimos a raiz do projeto (`ROOT`), `sys.path` e carregamos `data_processed/pede_unificado.parquet` (coluna **`fase`** como nível inteiro **0–8**: Alfa→0; `1`/`1A`/`1B`→1; … até 8, via `apply_fase_column_normalization`).
- Se o parquet ainda não existir, `ensure_parquet` em `src/pede_model.py` chama `build_unified` em `src/pede_cleaning.py` (mesmo fluxo do app).
- A **seção 2** mostra explicitamente a limpeza e a unificação a partir dos CSVs; depois dela, o `df` usado no restante do notebook é o recém-gerado.
- A **seção 3** consolida a **análise exploratória** (correlações, evolução por ano do painel, padrões de defasagem) — material direto para **slides** antes do modelo.


In [65]:
from pathlib import Path
import sys

# Raiz do projeto (pasta com src/ e data_processed/)
ROOT = Path.cwd()
if (ROOT / "src" / "pede_model.py").exists():
    pass
elif (ROOT.parent / "src" / "pede_model.py").exists():
    ROOT = ROOT.parent
else:
    ROOT = Path("..").resolve()

sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

import pandas as pd
from src.pede_model import ensure_parquet
from src.pede_cleaning import apply_fase_column_normalization

pq = ensure_parquet(ROOT)
df = apply_fase_column_normalization(pd.read_parquet(pq))
print("Parquet:", pq)
print("Shape:", df.shape)
df.head()


ROOT = c:\Users\rhomu\OneDrive\Área de Trabalho\Pós Tech - Data Analytics\Grupo16TechChallenge\Fase_05\Fase05_10DTAT_Grupo48
Parquet: c:\Users\rhomu\OneDrive\Área de Trabalho\Pós Tech - Data Analytics\Grupo16TechChallenge\Fase_05\Fase05_10DTAT_Grupo48\data_processed\pede_unificado.parquet
Shape: (3030, 44)


,ra,ano_cohorte,fase,turma,nome,ano_nascimento,data_nasc,idade_referencia,genero,ano_ingresso,...,por,ing,indicado,atingiu_pv,fase_ideal,defasagem,rec_psicologia,destaque_ieg,destaque_ida,destaque_ipv
0,RA-1,2022,7,A,Aluno-1,2003.0,NaT,19.0,Feminino,2016.0,...,3.5,6.0,Sim,Não,Fase 8 (Universitários),-1.0,Requer avaliação,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
1,RA-2,2022,7,A,Aluno-2,2005.0,NaT,17.0,Feminino,2017.0,...,4.5,9.7,Não,Não,Fase 7 (3º EM),0.0,Sem limitações,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
2,RA-3,2022,7,A,Aluno-3,2005.0,NaT,17.0,Feminino,2016.0,...,4.0,6.9,Não,Não,Fase 7 (3º EM),0.0,Sem limitações,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Destaque: A sua boa integração aos Princípios ...
3,RA-4,2022,7,A,Aluno-4,2005.0,NaT,17.0,Masculino,2017.0,...,3.5,8.7,Não,Não,Fase 7 (3º EM),0.0,Requer avaliação,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
4,RA-5,2022,7,A,Aluno-5,2005.0,NaT,17.0,Feminino,2016.0,...,2.9,5.7,Não,Não,Fase 7 (3º EM),0.0,Requer avaliação,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...


## 2. Limpeza e unificação PEDE (2022–2024)

Esta seção reproduz o fluxo de limpeza/unificação usando **`src/pede_cleaning.py`**, que:

- remove colunas duplicadas do CSV (Destaque IPV; Ativo/Inativo);
- harmoniza nomes e tipos;
- empilha os três anos com chave `(ra, ano_cohorte)`;
- grava `data_processed/pede_unificado.parquet`.

**Requisito:** execute antes a **seção 1** (variável `ROOT` e `sys.path`). O `df` produzido aqui substitui o da célula anterior e alimenta a **seção 3 (EDA)** e as etapas seguintes de modelagem.

In [66]:
from src.pede_cleaning import build_unified, cleaning_report

OUT_PARQUET = ROOT / "data_processed" / "pede_unificado.parquet"
df = build_unified(root=ROOT, save_parquet=OUT_PARQUET)
cleaning_report(df)

{'n_rows': 3030,
 'n_ra_distintos': 1661,
 'linhas_por_ano': {2022: 860, 2023: 1014, 2024: 1156},
 'cols': ['ra',
  'ano_cohorte',
  'fase',
  'turma',
  'nome',
  'ano_nascimento',
  'data_nasc',
  'idade_referencia',
  'genero',
  'ano_ingresso',
  'instituicao_ensino',
  'escola',
  'status_aluno',
  'pedra_20',
  'pedra_21',
  'pedra_22',
  'pedra_23',
  'pedra_atual',
  'inde_cohorte',
  'inde_hist_22',
  'inde_hist_23',
  'inde_hist_24',
  'cg',
  'cf',
  'ct',
  'n_avaliacoes',
  'iaa',
  'ieg',
  'ips',
  'ida',
  'ipv',
  'ian',
  'ipp',
  'mat',
  'por',
  'ing',
  'indicado',
  'atingiu_pv',
  'fase_ideal',
  'defasagem',
  'rec_psicologia',
  'destaque_ieg',
  'destaque_ida',
  'destaque_ipv'],
 'na_rate_inde_cohorte': 0.06105610561056106,
 'na_rate_ian': 0.0,
 'na_rate_fase': 0.012541254125412541}

In [67]:
df.head(10)

,ra,ano_cohorte,fase,turma,nome,ano_nascimento,data_nasc,idade_referencia,genero,ano_ingresso,...,por,ing,indicado,atingiu_pv,fase_ideal,defasagem,rec_psicologia,destaque_ieg,destaque_ida,destaque_ipv
0,RA-1,2022,7,A,Aluno-1,2003.0,NaT,19.0,Feminino,2016.0,...,3.5,6.0,Sim,Não,Fase 8 (Universitários),-1.0,Requer avaliação,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
1,RA-2,2022,7,A,Aluno-2,2005.0,NaT,17.0,Feminino,2017.0,...,4.5,9.7,Não,Não,Fase 7 (3º EM),0.0,Sem limitações,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
2,RA-3,2022,7,A,Aluno-3,2005.0,NaT,17.0,Feminino,2016.0,...,4.0,6.9,Não,Não,Fase 7 (3º EM),0.0,Sem limitações,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Destaque: A sua boa integração aos Princípios ...
3,RA-4,2022,7,A,Aluno-4,2005.0,NaT,17.0,Masculino,2017.0,...,3.5,8.7,Não,Não,Fase 7 (3º EM),0.0,Requer avaliação,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
4,RA-5,2022,7,A,Aluno-5,2005.0,NaT,17.0,Feminino,2016.0,...,2.9,5.7,Não,Não,Fase 7 (3º EM),0.0,Requer avaliação,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
5,RA-6,2022,7,A,Aluno-6,2004.0,NaT,18.0,Feminino,2021.0,...,5.3,2.3,Sim,Não,Fase 8 (Universitários),-1.0,Sem limitações,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
6,RA-7,2022,7,A,Aluno-7,2004.0,NaT,18.0,Masculino,2017.0,...,5.7,9.0,Não,Não,Fase 8 (Universitários),-1.0,Sem limitações,Destaque: A sua boa entrega das lições de casa.,Destaque: As suas boas notas na Passos Mágicos.,Destaque: A sua boa integração aos Princípios ...
7,RA-8,2022,7,A,Aluno-8,2002.0,NaT,20.0,Feminino,2018.0,...,0.7,2.9,Não,Não,Fase 8 (Universitários),-1.0,Requer avaliação,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
8,RA-9,2022,7,A,Aluno-9,2004.0,NaT,18.0,Feminino,2019.0,...,6.0,8.7,Sim,Sim,Fase 8 (Universitários),-1.0,Sem limitações,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Destaque: A sua boa integração aos Princípios ...
9,RA-10,2022,7,A,Aluno-10,2004.0,NaT,18.0,Feminino,2021.0,...,2.6,6.4,Não,Não,Fase 8 (Universitários),-1.0,Requer avaliação,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...


In [68]:
dup = df.duplicated(subset=["ra", "ano_cohorte"])
assert not dup.any(), f"Duplicatas: {dup.sum()}"
print("OK: chave (ra, ano_cohorte) única.")

OK: chave (ra, ano_cohorte) única.


## 3. Análise exploratória (EDA)

Conteúdo alinhado aos tópicos de **slides** da apresentação:

- **Correlações:** matriz de Pearson entre os principais indicadores numéricos da base unificada.
- **Evolução dos indicadores:** médias por **`ano_cohorte`** (2022, 2023, 2024 — cada valor é uma “foto” do PEDE no painel; o CSV **não** traz evolução dentro do mesmo ano civil).
- **Identificação de padrões:** distribuição de **`defasagem`** e taxa de **`defasagem < 0`** por ano (alinhado ao alvo do modelo: fase efetiva abaixo da ideal, conforme a própria base).

**Ressalva:** correlações medem **associação linear**, não causalidade. Faixas oficiais de IAN (10 / 5 / 2,5) e pesos do INDE por fase estão em `pede_pontos_importantes.md`.

In [69]:
# Correlações (Pearson) entre variáveis numéricas principais
import plotly.express as px

eda_cols = [
    c
    for c in [
        "inde_cohorte",
        "ian",
        "ida",
        "ieg",
        "iaa",
        "ips",
        "ipp",
        "ipv",
        "mat",
        "por",
        "ing",
        "defasagem",
        "idade_referencia",
        "fase",
    ]
    if c in df.columns
]
R = df[eda_cols].corr(numeric_only=True)
fig_corr = px.imshow(
    R,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Matriz de correlação de Pearson (base harmonizada)",
)
fig_corr.show()

In [70]:
# Evolução dos indicadores (média por ano do painel não há granularidade mensal no CSV)
import plotly.express as px

cols_ev = [c for c in ["inde_cohorte", "ida", "ieg", "ian", "ips", "ipv", "iaa"] if c in df.columns]
by = df.groupby("ano_cohorte", observed=True)[cols_ev].mean().reset_index()
long = by.melt(id_vars="ano_cohorte", var_name="indicador", value_name="media")
fig_evo = px.line(
    long,
    x="ano_cohorte",
    y="media",
    color="indicador",
    markers=True,
    title="Evolução agregada: média dos indicadores por ano do painel",
    labels={"ano_cohorte": "Ano do painel", "media": "Média"},
)
fig_evo.update_layout(xaxis=dict(dtick=1))
fig_evo.show()

In [71]:
# Padrões: defasagem e taxa de defasagem negativa por ano do painel
import plotly.express as px

ev = (
    df.assign(defas_neg=(df["defasagem"] < 0).astype(int))
    .groupby("ano_cohorte", as_index=False)["defas_neg"]
    .mean()
)
fig_taxa = px.bar(
    ev,
    x="ano_cohorte",
    y="defas_neg",
    title="Taxa de defasagem negativa (defasagem < 0) por ano do painel",
    labels={"defas_neg": "Proporção", "ano_cohorte": "Ano do painel"},
)
fig_taxa.update_layout(xaxis=dict(dtick=1))
fig_taxa.show()

fig_hist = px.histogram(
    df.dropna(subset=["defasagem"]),
    x="defasagem",
    color="ano_cohorte",
    nbins=45,
    barmode="overlay",
    opacity=0.55,
    title="Distribuição de defasagem (D) por ano do painel",
    labels={"defasagem": "Defasagem (D)", "count": "Frequência"},
)
fig_hist.show()

## 4. Engenharia de atributos (feature engineering)

Implementado em `build_xy` e em `make_pipeline` em **`src/pede_model.py`**:

| Etapa | Descrição |
|-------|-----------|
| **Alvo** | `y = 1` se `defasagem < 0`, senão `0` |
| **Features** | `NUMERIC_FEATURES` + `genero` **não** usamos `ian` nem `defasagem` como preditores (evita vazamento do alvo) |
| **Categórica** | `genero` como string; vazios viram `"Desconhecido"` |
| **Filtro** | Mantemos linhas com **pelo menos um** indicador numérico não nulo |
| **Pré-processamento** | `ColumnTransformer`: mediana + `StandardScaler` nos numéricos; moda + `OneHotEncoder` em `genero` |


In [72]:
import numpy as np
from src.pede_model import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    build_xy,
)

X, y = build_xy(df)
mask = X[NUMERIC_FEATURES].notna().any(axis=1)
X = X.loc[mask]
y = y[mask.values]

print("Número de features numéricas:", len(NUMERIC_FEATURES))
print("Features numéricas:", NUMERIC_FEATURES)
print("Features categóricas:", CATEGORICAL_FEATURES)
print("Taxa de positivos (defasagem menor que zero):", float(y.mean()))
X.describe(include="all").T.head(20)


Número de features numéricas: 19
Features numéricas: ['idade_referencia', 'fase', 'ano_ingresso', 'cg', 'cf', 'ct', 'n_avaliacoes', 'iaa', 'ieg', 'ips', 'ipp', 'ida', 'mat', 'por', 'ing', 'ipv', 'inde_hist_22', 'inde_hist_23', 'ano_cohorte']
Features categóricas: ['genero']
Taxa de positivos (defasagem menor que zero): 0.5567656765676567


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
idade_referencia,2631.0,NaN,NaN,NaN,12.548081,3.283307,7.0,10.0,12.0,15.0,27.0
fase,2992.0,<NA>,<NA>,<NA>,2.475602,2.148781,0.0,1.0,2.0,4.0,8.0
ano_ingresso,3030.0,NaN,NaN,NaN,2021.563696,1.822171,2016.0,2021.0,2022.0,2023.0,2024.0
cg,860.0,NaN,NaN,NaN,430.516279,248.432761,1.0,215.75,430.5,645.25,862.0
cf,860.0,NaN,NaN,NaN,75.519767,52.31267,1.0,30.0,67.0,118.0,192.0
ct,860.0,NaN,NaN,NaN,6.598837,3.975858,1.0,3.0,6.0,9.0,18.0
n_avaliacoes,2954.0,NaN,NaN,NaN,3.031821,1.06367,0.0,2.0,3.0,4.0,6.0
iaa,2865.0,NaN,NaN,NaN,7.919476,2.627066,0.0,7.9,8.8,9.5,10.0
ieg,2954.0,NaN,NaN,NaN,7.945565,2.151814,0.0,7.3,8.6,9.4,10.0
ips,2846.0,NaN,NaN,NaN,6.296954,1.78421,2.5,5.02,7.5,7.5,10.0


## 5. Separação em treino e teste

Holdout **25%** para teste, `random_state=42`, amostragem **estratificada** por `y` igual a `train_test_split` em `train_risk_model` em **`src/pede_model.py`**.


In [73]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Treino:", X_train.shape[0], "| Teste:", X_test.shape[0])
print("Positivos — treino:", float(y_train.mean()), "| teste:", float(y_test.mean()))


Treino: 2272 | Teste: 758
Positivos — treino: 0.5567781690140845 | teste: 0.5567282321899736


## 6. Modelagem preditiva

`RandomForestClassifier` dentro de um **`sklearn.pipeline.Pipeline`** com pré-processamento (`make_pipeline` em **`src/pede_model.py`**): `n_estimators=400`, `max_depth=14`, `class_weight='balanced_subsample'`, etc.


In [74]:
from src.pede_model import make_pipeline

pipe = make_pipeline()
pipe.fit(X_train, y_train)
print("Treinamento concluído.")
pipe


Treinamento concluído.


,steps,"[('prep', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 7. Avaliação dos resultados

- **ROC-AUC** no conjunto de teste (probabilidade da classe positiva).
- **Relatório de classificação** e **matriz de confusão** com limiar 0,5.
- Gráficos com **Plotly** (dependência do projeto em `requirements.txt`).


In [75]:
from sklearn.metrics import auc, classification_report, confusion_matrix, roc_auc_score, roc_curve
import plotly.graph_objects as go

proba = pipe.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

roc = float(roc_auc_score(y_test, proba))
print("ROC-AUC (teste):", roc)
print()
print(classification_report(y_test, pred, digits=3))

cm = confusion_matrix(y_test, pred)
print("Matriz de confusão [[TN FP] na primeira linha, [FN TP] na segunda]:")
print(cm)

fpr, tpr, _ = roc_curve(y_test, proba)
roc_auc_curve = auc(fpr, tpr)

fig_cm = go.Figure(
    data=go.Heatmap(
        z=cm[::-1],
        x=["Predito 0", "Predito 1"],
        y=["Real 1", "Real 0"],
        text=cm[::-1],
        texttemplate="%{text}",
        colorscale="Blues",
        showscale=True,
    )
)
fig_cm.update_layout(
    title="Matriz de confusão (teste)",
    xaxis_title="Predito",
    yaxis_title="Real",
    width=520,
    height=420,
)
fig_cm.show()

fig_roc = go.Figure()
fig_roc.add_trace(
    go.Scatter(
        x=fpr,
        y=tpr,
        mode="lines",
        name=f"ROC (AUC = {roc_auc_curve:.3f})",
        fill="tozeroy",
        fillcolor="rgba(99,110,250,0.15)",
    )
)
fig_roc.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Aleatório", line=dict(dash="dash", color="gray"))
)
fig_roc.update_layout(
    title="Curva ROC — conjunto de teste",
    xaxis_title="Taxa de falsos positivos",
    yaxis_title="Taxa de verdadeiros positivos",
    width=560,
    height=480,
    yaxis=dict(scaleanchor="x", scaleratio=1),
)
fig_roc.show()


ROC-AUC (teste): 0.9262370232453171

              precision    recall  f1-score   support

           0      0.839     0.774     0.805       336
           1      0.830     0.882     0.855       422

    accuracy                          0.834       758
   macro avg      0.835     0.828     0.830       758
weighted avg      0.834     0.834     0.833       758

Matriz de confusão [[TN FP] na primeira linha, [FN TP] na segunda]:
[[260  76]
 [ 50 372]]


### 7.1 Importância das variáveis (Random Forest)

Após o `OneHotEncoder`, os nomes das colunas transformadas vêm de `get_feature_names_out()` do `ColumnTransformer` no pipeline.


In [76]:
import pandas as pd
import plotly.graph_objects as go

prep = pipe.named_steps["prep"]
clf = pipe.named_steps["clf"]
names = prep.get_feature_names_out()
imps = clf.feature_importances_
imp_df = pd.DataFrame({"feature": names, "importance": imps}).sort_values("importance", ascending=False).head(25)

fig_imp = go.Figure(
    go.Bar(x=imp_df["importance"], y=imp_df["feature"], orientation="h", marker_color="#636EFA")
)
fig_imp.update_layout(
    title="Top 25 importâncias (espaço transformado pelo pipeline)",
    xaxis_title="Importância",
    yaxis_title="",
    height=700,
    margin=dict(l=160, r=24, t=60, b=48),
)
fig_imp.show()
imp_df


,feature,importance
0,num__idade_referencia,0.195011
1,num__fase,0.128008
16,num__inde_hist_22,0.104690
15,num__ipv,0.055350
8,num__ieg,0.050122
4,num__cf,0.041698
11,num__ida,0.041258
12,num__mat,0.040715
3,num__cg,0.040069
13,num__por,0.039349


## 8. Principais insights (para slides)

Três mensagens centrais deste projeto, com **gráficos de apoio** na célula seguinte.

### Insight 1 — O que dá para ver no tempo (e o que não dá)

A base é um **painel** (`RA` × **`ano_cohorte`**: 2022, 2023, 2024). **Não há** granularidade mensal/bimestral no CSV. Mesmo assim, dá para acompanhar **tendências agregadas** (ex.: taxa de **`defasagem < 0`** e médias de indicadores por ano), úteis para narrativa sempre com a ressalva de que **correlação e tendência agregada não implicam causalidade** nem impacto isolado do programa.

### Insight 2 — IAA é escuta; IDA é resultado

No agregado, a associação linear **IAA × IDA** costuma ser **fraca**: a autoavaliação captura **percepção** do aluno, mas **não substitui** o desempenho medido (IDA, notas). Isso reforça uso do IAA em **acolhimento e diagnóstico**, e do IDA/IEG em **plano pedagógico**.

### Insight 3 — Triagem de risco com ML (sem “colar” no alvo)

O modelo estima **P(`defasagem` < 0)** com **Random Forest**, **sem** colocar **`ian`** nem **`defasagem`** nas *features* (evita vazamento direto do critério de defasagem). A **curva ROC** no holdout resume a capacidade de **ordenar risco** no conjunto de teste ferramenta de **triagem**, não substituto de decisão humana.

**Pré-requisito:** execute as seções **6 e 7** antes (variáveis `pipe`, `X_test`, `y_test`) e mantenha o `df` carregado.

In [77]:
# Gráficos dos 3 insights (slides) — requer: df, pipe, X_test, y_test
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import roc_auc_score, roc_curve

# --- Insight 1: tendência agregada por ano do painel ---
ev = (
    df.assign(defas_neg=(df["defasagem"] < 0).astype(int))
    .groupby("ano_cohorte", as_index=False)["defas_neg"]
    .mean()
)
fig_ins1 = px.bar(
    ev,
    x="ano_cohorte",
    y="defas_neg",
    title="Insight 1 — Taxa de defasagem negativa por ano do painel",
    labels={"defas_neg": "Proporção (defasagem < 0)", "ano_cohorte": "Ano do painel"},
)
fig_ins1.update_layout(xaxis=dict(dtick=1))
fig_ins1.show()

# --- Insight 2: IAA vs IDA (percepção vs desempenho) ---
d2 = df.dropna(subset=["iaa", "ida"]).copy()
r_iaa_ida = float(d2["iaa"].corr(d2["ida"]))
sample = d2.sample(min(1500, len(d2)), random_state=7)
fig_ins2 = px.scatter(
    sample,
    x="ida",
    y="iaa",
    color="ano_cohorte",
    title=f"Insight 2 — IAA × IDA (amostra; r ≈ {r_iaa_ida:.2f})",
    labels={"ida": "IDA", "iaa": "IAA", "ano_cohorte": "Ano"},
    opacity=0.55,
)
fig_ins2.show()

# --- Insight 3: ROC no teste (triagem de risco) ---
proba_te = pipe.predict_proba(X_test)[:, 1]
roc_te = float(roc_auc_score(y_test, proba_te))
fpr, tpr, _ = roc_curve(y_test, proba_te)
fig_ins3 = go.Figure()
fig_ins3.add_trace(
    go.Scatter(
        x=fpr,
        y=tpr,
        mode="lines",
        name=f"Modelo (AUC = {roc_te:.3f})",
        fill="tozeroy",
        fillcolor="rgba(99,110,250,0.15)",
    )
)
fig_ins3.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Aleatório", line=dict(dash="dash", color="gray")))
fig_ins3.update_layout(
    title="Insight 3 — Curva ROC no conjunto de teste (P(defasagem < 0))",
    xaxis_title="Taxa de falsos positivos",
    yaxis_title="Taxa de verdadeiros positivos",
    width=560,
    height=480,
    yaxis=dict(scaleanchor="x", scaleratio=1),
)
fig_ins3.show()

print(f"Resumo numérico — correlação IAA×IDA (toda a base, válidos): {r_iaa_ida:.3f} | ROC-AUC (teste): {roc_te:.3f}")

Resumo numérico — correlação IAA×IDA (toda a base, válidos): 0.116 | ROC-AUC (teste): 0.926
